# 🔬 Chaos 2 Clarity — Notebook 04: Experiment Runner

Runs all 8 experiments from the paper. Each section is self-contained.

**Prerequisites:** Run `00_data_generation.ipynb` first.

## Setup

In [ ]:
!pip install -q duckdb pandas numpy chromadb google-generativeai openai scipy statsmodels scikit-learn matplotlib seaborn tqdm networkx sentence-transformers faker

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import sys, os, json, time
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

PROJECT_DIR = '/content/drive/MyDrive/chaos2clarity'

# Add source to path
sys.path.insert(0, '/content/chaos2clarity')  # or wherever the repo is cloned

# ── API Keys (stored as Colab Secrets) ──
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

# ── Initialize LLM Clients ──
import google.generativeai as genai
genai.configure(api_key=GOOGLE_API_KEY)
gemini = genai.GenerativeModel('gemini-1.5-pro')

from openai import OpenAI
openai_client = OpenAI(api_key=OPENAI_API_KEY)

print('✅ LLM clients initialized')

In [ ]:
# ── Load Data Environment ──
import duckdb
conn = duckdb.connect(f'{PROJECT_DIR}/data/retail.duckdb', read_only=False)

from src.data_generator import get_schema_ddl, get_column_descriptions_str
schema_ddl = get_schema_ddl(conn)
col_descriptions = get_column_descriptions_str()

# ── Load Questions ──
from src.eval_harness import load_questions
questions = load_questions(f'{PROJECT_DIR}/eval/questions')
print(f'✅ Loaded {len(questions)} questions')

# ── Build Semantic Model ──
from src.semantic_layer import build_gold_semantic_model, synthesize_semantic_model
# Use gold model as starting point (in practice, this would be auto-synthesized)
semantic_model = build_gold_semantic_model()
print(f'✅ Semantic model: {len(semantic_model.entities)} entities, {len(semantic_model.metrics)} metrics')

In [ ]:
# ── Initialize Systems ──
from src.vector_store import VectorStore
from src.feedback_loop import FeedbackLoop
from src.orchestration import (
    C2CPipeline, C2CMonolithic, C2CNoPlanner,
    C2CNoValidator, C2CNoRetry, C2CNoVector, C2CNoFeedback
)
from src.baselines import B1DirectLLM, B2SchemaAwareLLM, B3PipelineNoSem
from src.eval_harness import evaluate_system, compute_metrics, save_results, compute_degradation, compute_metrics_by_tier
from src.stats import mcnemar_test, mann_whitney_latency, run_all_significance_tests

# Rate limiting helper
import time as _time
RATE_LIMIT_DELAY = 1.0  # seconds between LLM calls (adjust for your tier)

def rate_limited_system(system_fn, delay=RATE_LIMIT_DELAY):
    """Wrap a system function with rate limiting."""
    def wrapper(nl_prompt):
        _time.sleep(delay)
        return system_fn(nl_prompt)
    return wrapper

print('✅ All systems ready')

---
## 🔴 Experiment 1: Baseline vs. C2C (Central Hypothesis)

**Claim:** EA(C2C-Full) ≥ EA(𝔅₁) + 25pp; L3 EA(C2C) > 0% while EA(𝔅₁) = 0%

**Systems:** B1-Direct, B2-SchemaAware, C2C-Full

**Queries:** 50 × 3 = 150 LLM calls

In [ ]:
# ── Initialize Experiment 1 Systems ──
b1 = B1DirectLLM(openai_client, conn, schema_ddl)
b2 = B2SchemaAwareLLM(openai_client, conn, schema_ddl, col_descriptions)

# C2C-Full: fresh vector store + fresh feedback loop
vs_e1 = VectorStore(collection_name='c2c_exp1')
fl_e1 = FeedbackLoop(alpha=0.15)
from src.semantic_layer import build_gold_semantic_model
sm_e1 = build_gold_semantic_model()  # frozen snapshot for E1
c2c_full = C2CPipeline(gemini, conn, sm_e1, vs_e1, fl_e1, schema_ddl, max_retries=3)

exp1_systems = {
    'B1-Direct': rate_limited_system(b1),
    'B2-SchemaAware': rate_limited_system(b2),
    'C2C-Full': rate_limited_system(c2c_full),
}

print('✅ Experiment 1 systems initialized')

In [ ]:
# ── Run Experiment 1 ──
exp1_results = {}
exp1_metrics = {}

for sys_name, sys_fn in exp1_systems.items():
    print(f'\n▶ Running {sys_name} on {len(questions)} questions...')
    results = evaluate_system(sys_name, questions, sys_fn)
    exp1_results[sys_name] = results
    exp1_metrics[sys_name] = compute_metrics(results)
    save_results(results, f'{PROJECT_DIR}/eval/results/exp1_{sys_name}.json')
    print(f'  EA: {exp1_metrics[sys_name]["EA"]*100:.1f}%  RC: {exp1_metrics[sys_name]["RC"]*100:.1f}%')

In [ ]:
# ── Experiment 1: Results Table (Paper Table 2) ──
print('\n' + '='*80)
print('TABLE 2: Main Results (Experiment 1)')
print('='*80)
print(f'{"System":<25} {"L1 EA":>8} {"L2 EA":>8} {"L3 EA":>8} {"L4 EA":>8} {"EA":>8} {"RC":>8}')
print('-'*80)
for sys_name in ['B1-Direct', 'B2-SchemaAware', 'C2C-Full']:
    m = exp1_metrics[sys_name]
    print(f'{sys_name:<25} {m.get("L1_EA",0)*100:>7.1f}% {m.get("L2_EA",0)*100:>7.1f}% {m.get("L3_EA",0)*100:>7.1f}% {m.get("L4_EA",0)*100:>7.1f}% {m["EA"]*100:>7.1f}% {m["RC"]*100:>7.1f}%')

# Delta
delta_ea = exp1_metrics['C2C-Full']['EA'] - exp1_metrics['B1-Direct']['EA']
print(f'\n📊 C2C vs B1 EA delta: {delta_ea*100:+.1f} pp')
print(f'   Central hypothesis (≥25pp): {"✅ PASS" if delta_ea >= 0.25 else "❌ FAIL"}')
print(f'   L3 cross-source: C2C={exp1_metrics["C2C-Full"].get("L3_EA",0)*100:.0f}% vs B1={exp1_metrics["B1-Direct"].get("L3_EA",0)*100:.0f}%')

In [ ]:
# ── Experiment 1: Statistical Significance ──
sig_tests = run_all_significance_tests(exp1_results, reference_system='C2C-Full')
for test_name, result in sig_tests.items():
    p = result.get('p_value', 1.0)
    sig = '✅' if result.get('significant', False) else '❌'
    print(f'{test_name}: p={p:.4f} {sig}')

# Save significance results
with open(f'{PROJECT_DIR}/eval/results/exp1_significance.json', 'w') as f:
    json.dump(sig_tests, f, indent=2, default=str)

---
## 🔴 Experiment 2: Semantic Layer Impact

**Claim:** Mechanism I (Automated Semantic Layer) is necessary for E1 suppression and E5 prevention.

**Systems:** B2-SchemaAware, B3-PipelineNoSem, C2C-Full

In [ ]:
# ── Initialize Experiment 2 ──
b3 = B3PipelineNoSem(gemini, conn, schema_ddl)

exp2_systems = {
    'B2-SchemaAware': rate_limited_system(b2),
    'B3-PipelineNoSem': rate_limited_system(b3),
    'C2C-Full': rate_limited_system(c2c_full),
}

# Run (reuse B2 and C2C from E1 if available)
exp2_results = {}
exp2_metrics = {}

if 'B2-SchemaAware' in exp1_results:
    exp2_results['B2-SchemaAware'] = exp1_results['B2-SchemaAware']
    exp2_metrics['B2-SchemaAware'] = exp1_metrics['B2-SchemaAware']
    print('♻️ Reusing B2 results from E1')

if 'C2C-Full' in exp1_results:
    exp2_results['C2C-Full'] = exp1_results['C2C-Full']
    exp2_metrics['C2C-Full'] = exp1_metrics['C2C-Full']
    print('♻️ Reusing C2C-Full results from E1')

# Only need to run B3
print(f'\n▶ Running B3-PipelineNoSem on {len(questions)} questions...')
exp2_results['B3-PipelineNoSem'] = evaluate_system('B3-PipelineNoSem', questions, exp2_systems['B3-PipelineNoSem'])
exp2_metrics['B3-PipelineNoSem'] = compute_metrics(exp2_results['B3-PipelineNoSem'])
save_results(exp2_results['B3-PipelineNoSem'], f'{PROJECT_DIR}/eval/results/exp2_B3.json')

In [ ]:
# ── Experiment 2: Semantic Synthesis Quality ──
from src.semantic_layer import synthesize_semantic_model, measure_synthesis_quality

# Run automated semantic synthesis
print('▶ Running automated semantic synthesis...')
auto_model = synthesize_semantic_model(conn, gemini, schema_ddl)
gold_model = build_gold_semantic_model()

synthesis_quality = measure_synthesis_quality(auto_model, gold_model)
print('\nTABLE 3: Semantic Synthesis Quality')
for k, v in synthesis_quality.items():
    print(f'  {k}: {v}')

with open(f'{PROJECT_DIR}/eval/results/exp2_synthesis_quality.json', 'w') as f:
    json.dump(synthesis_quality, f, indent=2)

In [ ]:
# ── Experiment 2: Error Class Rates ──
print('\nError Class Rates (Experiment 2):')
print(f'{"System":<25} {"E1":>6} {"E2":>6} {"E3":>6} {"E5":>6} {"L2 EA":>8} {"L3 EA":>8}')
print('-'*70)
for sys_name in ['B2-SchemaAware', 'B3-PipelineNoSem', 'C2C-Full']:
    m = exp2_metrics[sys_name]
    ed = m.get('error_distribution', {})
    print(f'{sys_name:<25} {ed.get("E1",0)*100:>5.1f}% {ed.get("E2",0)*100:>5.1f}% {ed.get("E3",0)*100:>5.1f}% {ed.get("E5",0)*100:>5.1f}% {m.get("L2_EA",0)*100:>7.1f}% {m.get("L3_EA",0)*100:>7.1f}%')

---
## 🔴 Experiment 3: Agent Ablation Study

**Claim:** Each pipeline stage contributes independently.

**Systems:** B2, ABL-Mono, ABL-NoPlanner, ABL-NoValidator, ABL-NoRetry, C2C-Full

**Queries:** 50 × 6 = 300 LLM calls (most expensive)

In [ ]:
# ── Initialize Experiment 3 Systems ──
sm_e3 = build_gold_semantic_model()
vs_e3 = VectorStore(collection_name='c2c_exp3'); vs_e3.clear()
fl_e3 = FeedbackLoop(alpha=0.15)

abl_mono = C2CMonolithic(gemini, conn, sm_e3, schema_ddl)
abl_noplanner = C2CNoPlanner(gemini, conn, sm_e3, vs_e3, fl_e3, schema_ddl)

fl_e3b = FeedbackLoop(alpha=0.15)
vs_e3b = VectorStore(collection_name='c2c_exp3b'); vs_e3b.clear()
abl_novalidator = C2CNoValidator(gemini, conn, sm_e3, vs_e3b, fl_e3b, schema_ddl)

fl_e3c = FeedbackLoop(alpha=0.15)
vs_e3c = VectorStore(collection_name='c2c_exp3c'); vs_e3c.clear()
abl_noretry = C2CNoRetry(gemini, conn, sm_e3, vs_e3c, fl_e3c, schema_ddl)

exp3_systems = {
    'B2-SchemaAware': rate_limited_system(b2),
    'ABL-Mono': rate_limited_system(abl_mono),
    'ABL-NoPlanner': rate_limited_system(abl_noplanner),
    'ABL-NoValidator': rate_limited_system(abl_novalidator),
    'ABL-NoRetry': rate_limited_system(abl_noretry),
    'C2C-Full': rate_limited_system(c2c_full),
}

print('✅ Experiment 3 systems initialized')

In [ ]:
# ── Run Experiment 3 ──
exp3_results = {}
exp3_metrics = {}

# Reuse B2 and C2C-Full from E1
for reuse_name in ['B2-SchemaAware', 'C2C-Full']:
    if reuse_name in exp1_results:
        exp3_results[reuse_name] = exp1_results[reuse_name]
        exp3_metrics[reuse_name] = exp1_metrics[reuse_name]
        print(f'♻️ Reusing {reuse_name} from E1')

# Run ablation variants
for sys_name in ['ABL-Mono', 'ABL-NoPlanner', 'ABL-NoValidator', 'ABL-NoRetry']:
    print(f'\n▶ Running {sys_name}...')
    results = evaluate_system(sys_name, questions, exp3_systems[sys_name])
    exp3_results[sys_name] = results
    exp3_metrics[sys_name] = compute_metrics(results)
    save_results(results, f'{PROJECT_DIR}/eval/results/exp3_{sys_name}.json')
    print(f'  EA: {exp3_metrics[sys_name]["EA"]*100:.1f}%  RC: {exp3_metrics[sys_name]["RC"]*100:.1f}%')

In [ ]:
# ── Experiment 3: Ablation Results (Paper Table 5) ──
print('\nTABLE 5: Ablation Results')
print(f'{"Variant":<25} {"EA":>8} {"RC":>8} {"E1":>6} {"E3":>6} {"P50ms":>8}')
print('-'*65)
for sys_name in ['B2-SchemaAware', 'ABL-Mono', 'ABL-NoPlanner', 'ABL-NoValidator', 'ABL-NoRetry', 'C2C-Full']:
    m = exp3_metrics.get(sys_name, {})
    ed = m.get('error_distribution', {})
    print(f'{sys_name:<25} {m.get("EA",0)*100:>7.1f}% {m.get("RC",0)*100:>7.1f}% {ed.get("E1",0)*100:>5.1f}% {ed.get("E3",0)*100:>5.1f}% {m.get("P50_ms",0):>7.0f}')

---
## 🔴 Experiment 4: Heterogeneous Data Handling (from E1 data)

In [ ]:
# ── Experiment 4: Compute Degradation (no new runs) ──
print('\nExperiment 4: Heterogeneous Data Degradation')
print(f'{"System":<25} {"Struct EA":>10} {"Hetero EA":>10} {"Degradation":>12}')
print('-'*65)
for sys_name in ['B1-Direct', 'B2-SchemaAware', 'C2C-Full']:
    tier_metrics = compute_metrics_by_tier(exp1_results[sys_name])
    deg = compute_degradation(tier_metrics)
    print(f'{sys_name:<25} {deg["structured_EA"]*100:>9.1f}% {deg["heterogeneous_EA"]*100:>9.1f}% {deg["absolute_degradation_pp"]:>10.1f}pp')

# Save
exp4_results = {}
for sys_name in ['B1-Direct', 'B2-SchemaAware', 'C2C-Full']:
    tier_metrics = compute_metrics_by_tier(exp1_results[sys_name])
    exp4_results[sys_name] = compute_degradation(tier_metrics)
with open(f'{PROJECT_DIR}/eval/results/exp4_degradation.json', 'w') as f:
    json.dump(exp4_results, f, indent=2)

---
## 🔴 Experiment 5: Feedback Learning Loop

**Claim:** Mechanism IV produces measurable quality improvement over successive batches.

**Setup:** 200 queries (50 × 4 batches), C2C-Full vs ABL-NoFeedback

In [ ]:
# ── Initialize Experiment 5 ──
# C2C-Full: fresh start
sm_e5_full = build_gold_semantic_model()
vs_e5_full = VectorStore(collection_name='c2c_exp5_full'); vs_e5_full.clear()
fl_e5_full = FeedbackLoop(alpha=0.15)
c2c_e5_full = C2CPipeline(gemini, conn, sm_e5_full, vs_e5_full, fl_e5_full, schema_ddl)

# ABL-NoFeedback: α=0, S frozen, V not updated
sm_e5_nofb = build_gold_semantic_model()
sm_e5_nofb.freeze()
vs_e5_nofb = VectorStore(collection_name='c2c_exp5_nofb'); vs_e5_nofb.clear()
fl_e5_nofb = FeedbackLoop(alpha=0.0)
fl_e5_nofb.disable()
c2c_e5_nofb = C2CNoFeedback(gemini, conn, sm_e5_nofb, vs_e5_nofb, fl_e5_nofb, schema_ddl)

# ABL-NoVector: S + δ active, V retrieval disabled (for Experiment 6)
sm_e5_novec = build_gold_semantic_model()
vs_e5_novec = VectorStore(collection_name='c2c_exp5_novec'); vs_e5_novec.clear()
fl_e5_novec = FeedbackLoop(alpha=0.15)
c2c_e5_novec = C2CNoVector(gemini, conn, sm_e5_novec, vs_e5_novec, fl_e5_novec, schema_ddl)

print('✅ Experiment 5+6 systems initialized (all start from empty V)')

In [ ]:
# ── Run Experiment 5+6: 200 queries × 3 arms ──
N_BATCHES = 4
exp5_checkpoints = {}
exp6_checkpoints = {}

exp5_6_systems = {
    'C2C-Full': c2c_e5_full,
    'ABL-NoFeedback': c2c_e5_nofb,
    'ABL-NoVector': c2c_e5_novec,
}

for batch_idx in range(N_BATCHES):
    batch_num = (batch_idx + 1) * 50
    print(f'\n{"="*60}')
    print(f'BATCH {batch_idx+1}/4 (T={batch_num})')
    print(f'{"="*60}')

    for sys_name, sys_fn in exp5_6_systems.items():
        print(f'  ▶ {sys_name}...')
        results = evaluate_system(sys_name, questions, rate_limited_system(sys_fn))
        metrics = compute_metrics(results)

        checkpoint_key = f'T{batch_num}'

        # Experiment 5 data
        if sys_name in ['C2C-Full', 'ABL-NoFeedback']:
            if sys_name not in exp5_checkpoints:
                exp5_checkpoints[sys_name] = {}
            exp5_checkpoints[sys_name][checkpoint_key] = {
                'EA': metrics['EA'],
                'RC': metrics['RC'],
                'E1_rate': metrics['error_distribution']['E1'],
                'first_pass_EA': metrics['first_pass_EA'],
            }

        # Experiment 6 data
        if sys_name in ['C2C-Full', 'ABL-NoVector']:
            if sys_name not in exp6_checkpoints:
                exp6_checkpoints[sys_name] = {}
            v_size = vs_e5_full.size() if sys_name == 'C2C-Full' else 0
            exp6_checkpoints[sys_name][checkpoint_key] = {
                'first_pass_EA': metrics['first_pass_EA'],
                'final_EA': metrics['EA'],
                'E1_rate': metrics['error_distribution']['E1'],
                'V_size': v_size,
            }

        print(f'    EA={metrics["EA"]*100:.1f}% E1={metrics["error_distribution"]["E1"]*100:.1f}%')

    # Apply feedback batch for C2C-Full only
    fb_result = c2c_e5_full.apply_feedback_batch()
    print(f'  ► Feedback batch applied: {fb_result}')

    # Also apply for ABL-NoVector (it has feedback, just no V retrieval)
    c2c_e5_novec.apply_feedback_batch()

# Save
with open(f'{PROJECT_DIR}/eval/results/exp5_learning_curve.json', 'w') as f:
    json.dump(exp5_checkpoints, f, indent=2)
with open(f'{PROJECT_DIR}/eval/results/exp6_vector_grounding.json', 'w') as f:
    json.dump(exp6_checkpoints, f, indent=2)

print('\n✅ Experiments 5+6 complete')

In [ ]:
# ── Experiment 5: Learning Curve Table ──
print('\nExperiment 5: Learning Curve')
print(f'{"Checkpoint":<12} {"C2C EA":>10} {"NoFB EA":>10} {"C2C E1":>8} {"NoFB E1":>8}')
print('-'*55)
for t in ['T50', 'T100', 'T150', 'T200']:
    c = exp5_checkpoints.get('C2C-Full', {}).get(t, {})
    n = exp5_checkpoints.get('ABL-NoFeedback', {}).get(t, {})
    print(f'{t:<12} {c.get("EA",0)*100:>9.1f}% {n.get("EA",0)*100:>9.1f}% {c.get("E1_rate",0)*100:>7.1f}% {n.get("E1_rate",0)*100:>7.1f}%')

---
## 🟡 Experiment 7: Error Taxonomy Distribution (from E1/E3 logs)

In [ ]:
# ── Experiment 7: Error Distribution (no new runs) ──
print('\nTABLE 4: Error Taxonomy Distribution')
print(f'{"System":<25} {"E1":>6} {"E2":>6} {"E3":>6} {"E4":>6} {"E5":>6} {"OK":>6}')
print('-'*65)
for sys_name in ['B1-Direct', 'B2-SchemaAware', 'ABL-Mono', 'C2C-Full']:
    results = None
    if sys_name in exp1_results:
        results = exp1_results[sys_name]
    elif sys_name in exp3_results:
        results = exp3_results[sys_name]
    if results:
        m = compute_metrics(results)
        ed = m.get('error_distribution', {})
        print(f'{sys_name:<25} {ed.get("E1",0)*100:>5.1f}% {ed.get("E2",0)*100:>5.1f}% {ed.get("E3",0)*100:>5.1f}% {ed.get("E4",0)*100:>5.1f}% {ed.get("E5",0)*100:>5.1f}% {ed.get("None",0)*100:>5.1f}%')

---
## 🟡 Experiment 8: Latency-Accuracy Tradeoff (from timing logs)

In [ ]:
# ── Experiment 8: Latency Table (no new runs) ──
print('\nTABLE 6: Latency-Accuracy Pareto')
print(f'{"System":<25} {"P50 ms":>10} {"P95 ms":>10} {"EA":>8} {"vs B1":>10}')
print('-'*70)

b1_p50 = exp1_metrics.get('B1-Direct', {}).get('P50_ms', 0)

for sys_name in ['B1-Direct', 'B2-SchemaAware', 'C2C-Full']:
    m = exp1_metrics.get(sys_name, {})
    p50 = m.get('P50_ms', 0)
    delta = p50 - b1_p50
    print(f'{sys_name:<25} {p50:>9.0f} {m.get("P95_ms",0):>9.0f} {m.get("EA",0)*100:>7.1f}% {delta:>+9.0f}ms')

# Add ablation variants if available
for sys_name in ['ABL-NoRetry', 'ABL-NoValidator']:
    m = exp3_metrics.get(sys_name, {})
    if m:
        p50 = m.get('P50_ms', 0)
        delta = p50 - b1_p50
        print(f'{sys_name:<25} {p50:>9.0f} {m.get("P95_ms",0):>9.0f} {m.get("EA",0)*100:>7.1f}% {delta:>+9.0f}ms')

---
## 📊 Generate Figures for Paper

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', font_scale=1.2)

# ── Figure: Learning Curve (Experiment 5) ──
fig, ax1 = plt.subplots(figsize=(10, 6))

checkpoints = [50, 100, 150, 200]
c2c_ea = [exp5_checkpoints.get('C2C-Full', {}).get(f'T{t}', {}).get('EA', 0) * 100 for t in checkpoints]
nofb_ea = [exp5_checkpoints.get('ABL-NoFeedback', {}).get(f'T{t}', {}).get('EA', 0) * 100 for t in checkpoints]
c2c_e1 = [exp5_checkpoints.get('C2C-Full', {}).get(f'T{t}', {}).get('E1_rate', 0) * 100 for t in checkpoints]
nofb_e1 = [exp5_checkpoints.get('ABL-NoFeedback', {}).get(f'T{t}', {}).get('E1_rate', 0) * 100 for t in checkpoints]

ax1.plot(checkpoints, c2c_ea, 'b-o', linewidth=2, markersize=8, label='C2C-Full EA')
ax1.plot(checkpoints, nofb_ea, 'b--s', linewidth=2, markersize=8, label='ABL-NoFeedback EA')
ax1.set_xlabel('Cumulative Queries Processed')
ax1.set_ylabel('Execution Accuracy (%)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

ax2 = ax1.twinx()
ax2.plot(checkpoints, c2c_e1, 'r-^', linewidth=2, markersize=8, label='C2C-Full E1 Rate')
ax2.plot(checkpoints, nofb_e1, 'r--v', linewidth=2, markersize=8, label='ABL-NoFeedback E1 Rate')
ax2.set_ylabel('E1 Error Rate (%)', color='red')
ax2.tick_params(axis='y', labelcolor='red')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')

plt.title('Experiment 5: Feedback Learning Loop')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/figures/exp5_learning_curve.pdf', dpi=300, bbox_inches='tight')
plt.savefig(f'{PROJECT_DIR}/figures/exp5_learning_curve.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Figure saved')

In [ ]:
# ── Figure: Error Taxonomy Stacked Bar (Experiment 7) ──
fig, ax = plt.subplots(figsize=(12, 6))

systems = ['B1-Direct', 'B2-SchemaAware', 'ABL-Mono', 'C2C-Full']
error_classes = ['E1', 'E2', 'E3', 'E4', 'E5']
colors = ['#e74c3c', '#f39c12', '#9b59b6', '#3498db', '#1abc9c']

data = {}
for sys_name in systems:
    results = exp1_results.get(sys_name) or exp3_results.get(sys_name)
    if results:
        m = compute_metrics(results)
        ed = m.get('error_distribution', {})
        data[sys_name] = [ed.get(ec, 0) * 100 for ec in error_classes]

x = np.arange(len(systems))
bottom = np.zeros(len(systems))

for i, ec in enumerate(error_classes):
    vals = [data.get(s, [0]*5)[i] for s in systems]
    ax.bar(x, vals, 0.6, bottom=bottom, label=ec, color=colors[i])
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels(systems, rotation=15)
ax.set_ylabel('Error Rate (%)')
ax.set_title('Experiment 7: Error Taxonomy Distribution')
ax.legend(title='Error Class')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/figures/exp7_error_taxonomy.pdf', dpi=300, bbox_inches='tight')
plt.savefig(f'{PROJECT_DIR}/figures/exp7_error_taxonomy.png', dpi=300, bbox_inches='tight')
plt.show()

---
## ✅ All Experiments Complete

Results saved to `eval/results/`. Ready to fill paper tables.

In [ ]:
# ── Final Summary ──
print('\n' + '='*80)
print('EXPERIMENT SUMMARY')
print('='*80)
print(f'E1 Baseline vs C2C:     ✅ Complete')
print(f'E2 Semantic Layer:      ✅ Complete')
print(f'E3 Agent Ablation:      ✅ Complete')
print(f'E4 Heterogeneous Data:  ✅ Complete')
print(f'E5 Feedback Loop:       ✅ Complete')
print(f'E6 Vector Grounding:    ✅ Complete')
print(f'E7 Error Taxonomy:      ✅ Complete')
print(f'E8 Latency-Accuracy:    ✅ Complete')
print(f'\nAll result files: {PROJECT_DIR}/eval/results/')
print(f'All figures: {PROJECT_DIR}/figures/')